In [1]:
import pandas as pd
import glob
import os
import numpy as np

# =========================
# 1. FUNÇÕES AUXILIARES
# =========================

def arquivo_mais_recente(padrao):
    arquivos = glob.glob(padrao)
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para o padrão: {padrao}")
    return max(arquivos, key=os.path.getctime)

def normalizar_texto(s):
    if s is None:
        return ""
    return str(s).strip().lower()

def encontrar_coluna(df, candidatos):
    cols_norm = {normalizar_texto(c): c for c in df.columns}
    for cand in candidatos:
        cand_norm = normalizar_texto(cand)
        if cand_norm in cols_norm:
            return cols_norm[cand_norm]

    # procura por ocorrência parcial
    for col in df.columns:
        col_norm = normalizar_texto(col)
        for cand in candidatos:
            cand_norm = normalizar_texto(cand)
            if cand_norm in col_norm:
                return col
    return None

def converter_numero_br(valor):
    """
    Converte valores como:
    '1.234', '1,234', '4,5', 'R$ 1.299,90', '1.299 avaliações'
    em float quando possível.
    """
    if pd.isna(valor):
        return np.nan

    v = str(valor).strip()

    if v == "":
        return np.nan

    # remove textos comuns
    remover = ["R$", "avaliações", "avaliação", "ratings", "rating", "opiniões", "opinião"]
    for r in remover:
        v = v.replace(r, "")

    v = v.strip()

    # mantém apenas números, ponto, vírgula e sinal
    permitido = "".join(ch for ch in v if ch.isdigit() or ch in [".", ",", "-"])
    v = permitido.strip()

    if v == "":
        return np.nan

    # heurística:
    # se tem ponto e vírgula -> formato BR de milhar/decimal
    if "." in v and "," in v:
        v = v.replace(".", "").replace(",", ".")
    else:
        # se só tem vírgula, tenta interpretar como decimal
        if "," in v:
            # se tiver mais de uma vírgula, remove tudo e mantém última como decimal
            if v.count(",") > 1:
                partes = v.split(",")
                v = "".join(partes[:-1]) + "." + partes[-1]
            else:
                v = v.replace(",", ".")
        # se só tem ponto, deixa como está

    try:
        return float(v)
    except:
        return np.nan

def resumo_dataset(df, nome_plataforma):
    # tenta achar colunas
    col_produto_id = encontrar_coluna(df, [
        "ASIN", "asin", "id_produto", "id produto", "produto_id", "mlb", "id"
    ])

    col_categoria = encontrar_coluna(df, [
        "Subcategoria", "subcategoria", "Categoria", "categoria"
    ])

    col_qtd_avaliacoes = encontrar_coluna(df, [
        "Qtd. Avaliações", "Qtd Avaliações", "Quantidade de Avaliações",
        "qtd avaliações", "qtd. avaliações", "avaliacoes", "avaliações",
        "total_avaliacoes", "total avaliações", "volume de avaliações"
    ])

    col_nota_produto = encontrar_coluna(df, [
        "Nota Geral", "nota geral", "Nota", "nota", "rating", "nota média"
    ])

    col_comentario = encontrar_coluna(df, [
        "Comentário", "comentario", "Review", "review", "Texto", "texto"
    ])

    # cópia de trabalho
    base = df.copy()

    # conversões numéricas
    if col_qtd_avaliacoes:
        base["_qtd_avaliacoes_num"] = base[col_qtd_avaliacoes].apply(converter_numero_br)
    else:
        base["_qtd_avaliacoes_num"] = np.nan

    if col_nota_produto:
        base["_nota_produto_num"] = base[col_nota_produto].apply(converter_numero_br)
    else:
        base["_nota_produto_num"] = np.nan

    # quantidade de produtos únicos
    if col_produto_id:
        qtd_produtos = base[col_produto_id].astype(str).str.strip().replace("", np.nan).dropna().nunique()
    else:
        qtd_produtos = np.nan

    # quantidade de categorias
    if col_categoria:
        qtd_categorias = base[col_categoria].astype(str).str.strip().replace("", np.nan).dropna().nunique()
    else:
        qtd_categorias = np.nan

    # quantidade total de avaliações (campo do produto)
    # para não duplicar produto por ter vários comentários, usa um registro por produto
    if col_produto_id and col_qtd_avaliacoes:
        produtos_unicos = base.dropna(subset=[col_produto_id]).copy()
        produtos_unicos[col_produto_id] = produtos_unicos[col_produto_id].astype(str).str.strip()
        produtos_unicos = produtos_unicos.drop_duplicates(subset=[col_produto_id])
        qtd_avaliacoes_total = produtos_unicos["_qtd_avaliacoes_num"].sum(skipna=True)
        media_avaliacoes_por_produto = produtos_unicos["_qtd_avaliacoes_num"].mean(skipna=True)
    else:
        qtd_avaliacoes_total = np.nan
        media_avaliacoes_por_produto = np.nan

    # comentários totais coletados
    if col_comentario:
        comentarios_validos = base[col_comentario].astype(str).str.strip()
        comentarios_totais = comentarios_validos.replace("", np.nan).dropna().shape[0]
    else:
        comentarios_totais = np.nan

    # média das notas dos produtos
    if col_produto_id and col_nota_produto:
        produtos_unicos_nota = base.dropna(subset=[col_produto_id]).copy()
        produtos_unicos_nota[col_produto_id] = produtos_unicos_nota[col_produto_id].astype(str).str.strip()
        produtos_unicos_nota = produtos_unicos_nota.drop_duplicates(subset=[col_produto_id])
        media_notas = produtos_unicos_nota["_nota_produto_num"].mean(skipna=True)
    else:
        media_notas = np.nan

    resumo = {
        "Plataforma": nome_plataforma,
        "Qtd. de produtos": int(qtd_produtos) if pd.notna(qtd_produtos) else None,
        "Qtd. de categorias": int(qtd_categorias) if pd.notna(qtd_categorias) else None,
        "Qtd. total de avaliações": round(float(qtd_avaliacoes_total), 2) if pd.notna(qtd_avaliacoes_total) else None,
        "Média de avaliações por produto": round(float(media_avaliacoes_por_produto), 2) if pd.notna(media_avaliacoes_por_produto) else None,
        "Comentários totais coletados": int(comentarios_totais) if pd.notna(comentarios_totais) else None,
        "Média de notas": round(float(media_notas), 2) if pd.notna(media_notas) else None,
    }

    colunas_identificadas = {
        "col_produto_id": col_produto_id,
        "col_categoria": col_categoria,
        "col_qtd_avaliacoes": col_qtd_avaliacoes,
        "col_nota_produto": col_nota_produto,
        "col_comentario": col_comentario
    }

    return resumo, colunas_identificadas

# =========================
# 2. LER OS ARQUIVOS MAIS RECENTES
# =========================

amazon_csv = arquivo_mais_recente("data/resultados_amazon/mais_vendidos_amazon_FULL_*.csv")
ml_csv = arquivo_mais_recente("data/resultados_ml/mais_vendidos_ml_FULL_*.csv")

print("Arquivo Amazon:", amazon_csv)
print("Arquivo Mercado Livre:", ml_csv)

amazon = pd.read_csv(amazon_csv, sep=";", encoding="utf-8-sig")
mercadolivre = pd.read_csv(ml_csv, sep=";", encoding="utf-8-sig")

print("\nColunas Amazon:")
print(list(amazon.columns))

print("\nColunas Mercado Livre:")
print(list(mercadolivre.columns))

# =========================
# 3. GERAR RESUMOS
# =========================

resumo_amazon, cols_amazon = resumo_dataset(amazon, "Amazon")
resumo_ml, cols_ml = resumo_dataset(mercadolivre, "Mercado Livre")

print("\nColunas identificadas - Amazon:")
print(cols_amazon)

print("\nColunas identificadas - Mercado Livre:")
print(cols_ml)

quadro_2 = pd.DataFrame([resumo_amazon, resumo_ml])

print("\nResumo para o Quadro 2:")
print(quadro_2)

# opcional: salvar
quadro_2.to_csv("quadro_2_resumo_dados.csv", sep=";", encoding="utf-8-sig", index=False)
print("\nArquivo salvo: quadro_2_resumo_dados.csv")

Arquivo Amazon: data/resultados_amazon\mais_vendidos_amazon_FULL_2026-02-d_07-31.csv
Arquivo Mercado Livre: data/resultados_ml\mais_vendidos_ml_FULL_2026-02-d_12-38.csv

Colunas Amazon:
['Posição Global', 'Posição Categoria', 'Subcategoria', 'ASIN', 'Nome', 'Preço', 'Nota Geral', 'Qtd. Avaliações', 'Características', 'País', 'Data Comentário', 'Nota Comentário', 'Comentário', 'Link']

Colunas Mercado Livre:
['Posição Global', 'Posição Categoria', 'Subcategoria', 'ASIN', 'Nome', 'Preço à vista', 'Nota Geral', 'Qtd. Avaliações', 'Caractéristicas', 'País', 'Nota Comentário', 'Data Comentário', 'Comentário', 'Link']

Colunas identificadas - Amazon:
{'col_produto_id': 'ASIN', 'col_categoria': 'Subcategoria', 'col_qtd_avaliacoes': 'Qtd. Avaliações', 'col_nota_produto': 'Nota Geral', 'col_comentario': 'Comentário'}

Colunas identificadas - Mercado Livre:
{'col_produto_id': 'ASIN', 'col_categoria': 'Subcategoria', 'col_qtd_avaliacoes': 'Qtd. Avaliações', 'col_nota_produto': 'Nota Geral', 'col_